## imports

In [1]:
import pandas as pd
import numpy as np
import joblib
import json

pd.set_option("display.max_columns", None)

## Load Model 1 n' 3 Artifact

In [2]:
popularity_df = pd.read_parquet(
    "../artifacts/popularity_recommender.parquet"
)

product_features = pd.read_parquet(
    "../artifacts/product_features.parquet"
)

feature_matrix = np.load(
    "../artifacts/product_feature_matrix.npy"
)

nn_model = joblib.load(
    "../artifacts/content_based_nn_model.pkl"
)

product_id_to_index = joblib.load(
    "../artifacts/product_id_to_index.pkl"
)

index_to_product_id = joblib.load(
    "../artifacts/index_to_product_id.pkl"
)

In [3]:
print("Popularity:", popularity_df.shape)
print("Products   :", product_features.shape)
print("Features   :", feature_matrix.shape)

Popularity: (4743, 4)
Products   : (32951, 6)
Features   : (32951, 78)


## Build Popularity Lookup

In [4]:
popularity_lookup = dict(
    zip(
        popularity_df["product_id"],
        popularity_df["popularity_score"]
    )
)

In [5]:
list(popularity_lookup.items())[:5]

[('aca2eb7d00ea1a7b8ebd4e68314663af', 1.0),
 ('422879e10f46682990de24d770e7f83d', 0.9042145593869731),
 ('99a4788cb24856965c36a24e339b6058', 0.9022988505747126),
 ('368c6c730842d78016ad823897a372db', 0.7260536398467433),
 ('389d119b48cf3043d311335e499d9c6b', 0.7049808429118773)]

## Content-Based Recommendation Function

In [6]:
def get_similar_products_with_scores(
    product_id,
    top_k=20
):
    
    product_idx = product_id_to_index[product_id]
    
    distances, indices = nn_model.kneighbors(
        feature_matrix[product_idx].reshape(1, -1),
        n_neighbors=top_k + 1
    )
    
    recommendations = []
    
    for dist, idx in zip(
        distances[0][1:],
        indices[0][1:]
    ):
        
        recommendations.append(
            {
                "product_id": index_to_product_id[idx],
                "similarity_score": 1 - dist
            }
        )
    
    return pd.DataFrame(recommendations)

# Hybrid Scoring Function

In [7]:
def hybrid_recommend(
    product_id,
    top_k=10,
    similarity_weight=0.7,
    popularity_weight=0.3
):

    recommendations = get_similar_products_with_scores(
        product_id,
        top_k=50      # get more candidates
    )

    # Keep only eligible products
    recommendations = recommendations[
        recommendations["product_id"].isin(
            popularity_lookup.keys()
        )
    ].copy()

    recommendations["popularity_score"] = (
        recommendations["product_id"]
        .map(popularity_lookup)
    )

    recommendations["hybrid_score"] = (
        similarity_weight
        * recommendations["similarity_score"]
        +
        popularity_weight
        * recommendations["popularity_score"]
    )

    recommendations = recommendations.sort_values(
        "hybrid_score",
        ascending=False
    )

    return recommendations.head(top_k)

## Test Hybrid Model

In [8]:
sample_product = product_features[
    "product_id"
].iloc[0]

sample_product

'1e9e8ef04dbcff4541ed26657ea517e5'

In [9]:
hybrid_results = hybrid_recommend(
    sample_product,
    top_k=10
)

hybrid_results

,product_id,similarity_score,popularity_score,hybrid_score
28,23ed7a10a36c6f88a757a9c83df9ea20,0.995686,0.026820,0.705026
0,6cd51331a07b84149502aa6c9c5b536e,0.999984,0.013410,0.704012
22,5a0c21ad6b82a1b61856e48ec959764d,0.996169,0.013410,0.701341
26,052b8660ee8a9ee18815d9b276694a10,0.995960,0.011494,0.700620
31,3488d2ce36e718097c1509444289ef7f,0.995599,0.009579,0.699793
35,ec1faa2edc27ce32372fee59d1f9e969,0.995212,0.009579,0.699522
45,cc1b5f3a64b4f452bdf5473f10dc6aaf,0.993147,0.009579,0.698076


In [10]:
hybrid_display = (
    hybrid_results
    .merge(
        product_features,
        on="product_id",
        how="left"
    )
)

hybrid_display[
    [
        "product_category_name",
        "similarity_score",
        "popularity_score",
        "hybrid_score"
    ]
]

,product_category_name,similarity_score,popularity_score,hybrid_score
0,perfumaria,0.995686,0.026820,0.705026
1,perfumaria,0.999984,0.013410,0.704012
2,perfumaria,0.996169,0.013410,0.701341
3,perfumaria,0.995960,0.011494,0.700620
4,perfumaria,0.995599,0.009579,0.699793
5,perfumaria,0.995212,0.009579,0.699522
6,perfumaria,0.993147,0.009579,0.698076


## Save Artifacts

In [11]:
model_4_metrics = {
    "model_type": "hybrid",
    "similarity_weight": 0.7,
    "popularity_weight": 0.3,
    "top_k": 10
}

with open(
    "../artifacts/model_4_metrics.json",
    "w"
) as f:
    json.dump(
        model_4_metrics,
        f,
        indent=4
    )

In [12]:
import joblib

hybrid_config = {
    "similarity_weight": 0.7,
    "popularity_weight": 0.3,
    "candidate_pool_size": 50
}

joblib.dump(
    hybrid_config,
    "../artifacts/hybrid_config.pkl"
)

['../artifacts/hybrid_config.pkl']

In [13]:
hybrid_results.head()

hybrid_display[
    [
        "product_category_name",
        "similarity_score",
        "popularity_score",
        "hybrid_score"
    ]
]

,product_category_name,similarity_score,popularity_score,hybrid_score
0,perfumaria,0.995686,0.026820,0.705026
1,perfumaria,0.999984,0.013410,0.704012
2,perfumaria,0.996169,0.013410,0.701341
3,perfumaria,0.995960,0.011494,0.700620
4,perfumaria,0.995599,0.009579,0.699793
5,perfumaria,0.995212,0.009579,0.699522
6,perfumaria,0.993147,0.009579,0.698076
